In [ ]:
"""
import the vowpal wabbit library.
"""
pip install vowpalwabbit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 11.9 MB/s eta 0:00:00


In [ ]:
"""
import the other required libraries.
"""
import pandas as pd
import vowpalwabbit
import numpy as np
import json

model ='gpt2'

In [ ]:
"""
load the genre_mapping dictioanry from the file directory. The genre_mapping dictionary is created once and used for all the three models. It contains the numerical representation of genres.
"""
genres_mappings = dict()
txt_file_path = f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/input/genre_mapping.txt'
with open(txt_file_path, 'r') as txt_file:
    genres_mappings = json.loads(txt_file.read())
print(genres_mappings)

unique_genres = list(genres_mappings.values())
print(unique_genres)

{'Crime': 1, 'Western': 2, 'Fantasy': 3, 'Science Fiction': 4, 'Comedy': 5, 'Family': 6, 'Mystery': 7, 'Drama': 8, 'Animation': 9, 'Thriller': 10, 'Documentary': 11, 'Horror': 12, 'Romance': 13, 'TV Movie': 14, 'History': 15, 'War': 16, 'Adventure': 17, 'Music': 18, 'Action': 19}
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]


In [ ]:
"""
load the dataset to be used for the experiment. This dataset was created by the data prepare step. This will act as the main dataset for this experiment also referred as the duplicate_instances_dataset in the report and will be refered as action_context_map for the experiment code.
"""
action_context_map = pd.read_csv(f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/data-prepare/{model}_action_context_map.csv')

In [ ]:
"""
convert the textual genres to the numerical genres columns.
"""
from ast import literal_eval
action_context_map['genres_indices'] = action_context_map['genres_indices'].apply(literal_eval)

In [ ]:
"""
print and see the records of action_context_map.
"""
action_context_map.head(100)

,genres_index,keywords_embedding_string,genres_indices,id,original_title_embedding_string,overview_embedding_string,release_date
0,1,7.77254701e-02 -5.38552813e-02 -5.79311967e-0...,"[11, 8]",1114,-6.28197715e-02 -2.74479061e-01 -3.25453132e-0...,-7.90620819e-02 1.27022997e-01 -7.71413565e-0...,6/29/2003
1,1,9.63690951e-02 6.21698722e-02 -5.16188264e-0...,[8],1115,1.28780708e-01 1.27590522e-01 -6.06716909e-0...,-1.14440762e-01 -7.02230632e-02 -2.19043717e-0...,9/15/2005
2,1,-1.05129234e-01 1.31294459e-01 -1.72948554e-0...,"[8, 16]",1116,-4.49490517e-01 -5.11738248e-02 7.86833838e-0...,-1.15996398e-01 -2.08327860e-01 -7.92207662e-0...,6/23/2006
3,1,-2.57868499e-01 6.26966059e-02 -7.99108744e-0...,"[19, 8, 10]",1123,2.59599864e-01 1.75047010e-01 -2.48356387e-0...,-2.46082172e-01 1.04989491e-01 -3.55010718e-0...,10/27/2006
4,1,2.70989593e-02 1.46407694e-01 -3.80637795e-0...,"[8, 7, 4]",1124,8.48557949e-02 1.20956279e-01 1.18412793e-0...,6.16751704e-03 -1.18230581e-01 -7.90594935e-01...,10/19/2006
...,...,...,...,...,...,...,...
95,1,-1.59929767e-01 2.12823581e-02 -3.19522738e-0...,"[13, 8]",1413,2.09821939e-01 1.21935057e-02 -7.08744526e-0...,-7.62087386e-03 3.83296236e-02 -4.92736667e-0...,10/1/1993
96,1,5.35899997e-02 2.90372372e-01 -6.34028554e-0...,"[8, 7, 13]",1414,1.68419197e-01 -5.64199686e-01 3.14390838e-0...,1.91535667e-01 3.17080855e-01 -9.22781467e-01...,9/4/1998
97,1,6.08246960e-02 1.66409895e-01 -7.16552436e-0...,"[5, 8, 1]",1415,-1.38422877e-01 -3.15836608e-01 -3.73536870e-0...,-1.81891635e-01 -1.12275474e-01 -1.17351137e-0...,9/5/2003
98,1,1.19114611e-02 2.95914233e-01 -6.46266282e-0...,"[8, 16]",1416,5.92958890e-02 -1.36997566e-01 1.62108451e-0...,-7.88879544e-02 1.85024276e-01 -6.03758574e-0...,11/22/2002


In [ ]:
"""
print and see the size of action_context_map
"""
action_context_map.shape[0]

190000

In [ ]:
"""
a variable to ensure test dataset is selected at random from the entire dataset.
"""
random_test_data_flag = True

In [ ]:
"""
the test dataset will always be the random test dataset. But incase if the experiment needs to be performed for a test data for which the model has prior knowledge, the set the flag to False in the above line.
"""
if random_test_data_flag:
  test_movies_df = pd.read_csv(f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/data-prepare/{model}_RANDOM_TEST_file.csv')
else:
  test_movies_df = pd.read_csv(f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/data-prepare/{model}_TEST_file.csv')
test_movies_df['genres_indices'] = test_movies_df['genres_indices'].apply(literal_eval)

In [ ]:
"""
print and see the records of test dataset.
"""
test_movies_df.head(10)

,keywords_embedding_string,genres_indices,id,original_title_embedding_string,release_date,overview_embedding_string
0,-2.11003348e-01 -7.42956027e-02 -3.51021618e-0...,[11],1223382,1.45722911e-01 -4.32809114e-01 -3.88971567e-0...,11/3/2023,-8.82894099e-02 5.96363544e-02 -6.09461248e-0...
1,1.10343076e-01 -4.42736521e-02 -6.94059968e-0...,[5],583413,9.50398296e-02 -4.76881087e-01 1.84726998e-0...,9/29/1984,2.55807310e-01 1.52284086e-01 -6.41608894e-01...
2,7.13114440e-03 -2.66596287e-01 -3.54553819e-0...,[11],342852,7.64546692e-02 -2.19964176e-01 -1.51549280e-0...,1/1/1968,1.30973354e-01 1.33584142e-01 -2.80212313e-01...
3,1.43736422e-01 -1.73615232e-01 -2.39788637e-0...,"[8, 5, 13]",10401,-1.26074955e-01 -2.28300750e-01 3.31108063e-0...,3/31/1999,-2.40932569e-01 1.21615514e-01 -4.67349231e-0...
4,1.84785947e-01 8.93511996e-02 -5.47323287e-0...,[11],802160,-9.86071602e-02 1.89811051e-01 -7.07017958e-0...,10/20/2000,3.14210355e-01 1.54408112e-01 -6.06957495e-01...
5,1.92347899e-01 1.01909772e-01 -2.49754086e-0...,"[4, 12]",1016490,1.09451003e-01 -4.37981868e-03 -3.74045044e-0...,NaN,4.39527817e-02 1.17761649e-01 -6.92441165e-01...
6,-1.58870101e-01 1.06403887e-01 -9.16866541e-0...,[11],331916,-1.14804186e-01 1.23109380e-02 -1.43504646e-0...,9/29/2012,2.45378330e-01 2.28965446e-01 -6.32845223e-01...
7,-2.20336676e-01 -3.86778980e-01 -8.47258091e-0...,[11],45558,-3.88191789e-01 -1.44735530e-01 -4.29979712e-0...,10/14/1992,-1.62216082e-01 -8.18149596e-02 2.65826564e-0...
8,1.26200512e-01 -1.86317027e-01 -4.67966795e-0...,"[12, 7]",37726,-4.89953607e-02 -3.45381796e-01 1.02610648e-0...,10/6/1998,-4.32252213e-02 3.45849134e-02 -2.66377777e-0...
9,-1.64095193e-01 2.56540626e-01 -9.53274190e-0...,[8],1142678,-4.94112037e-02 -4.24635679e-01 -3.83357167e-0...,3/2/2019,2.19030455e-01 1.69741988e-01 -6.08784378e-01...


In [ ]:
"""
print and see the size of action_context_map
"""
test_movies_df.shape[0]

38000

In [ ]:
"""
load the train dataset to be used for the experiment. This dataset was created by the data prepare step.
"""
train_movies_df = pd.read_csv(f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/data-prepare/{model}_TRAIN_file.csv')
train_movies_df['genres_indices'] = train_movies_df['genres_indices'].apply(literal_eval)

In [ ]:
"""
print and see the records of train dataset.
"""
train_movies_df.head(10)

,genres_index,keywords_embedding_string,genres_indices,id,original_title_embedding_string,overview_embedding_string,release_date,rewards,cost
0,1,2.09590763e-01 9.29589123e-02 -5.55353642e-0...,"[6, 9, 13, 3]",14128,-2.34530687e-01 -1.75846100e-01 -1.17273323e-0...,1.59452111e-02 -1.61797792e-01 -5.91270089e-01...,2/23/2002,0.0,1.0
1,5,1.79119170e-01 6.56483741e-03 -7.50915945e-0...,"[6, 3, 9, 14, 18, 7, 17]",13351,-7.54054785e-02 1.51387155e-01 -3.18139315e-0...,-3.81331854e-02 -3.62277962e-02 -2.75681019e-0...,9/3/1994,0.0,1.0
2,18,1.38275236e-01 6.01867028e-03 -3.08690369e-0...,"[8, 7, 10]",1126,-2.28183910e-01 -1.40631601e-01 -1.98941201e-0...,6.58507571e-02 1.50449738e-01 -2.34513208e-01...,10/7/2004,0.0,1.0
3,1,-3.84642512e-01 1.40074342e-01 -8.09253335e-0...,"[5, 12, 3]",16846,-5.91474473e-01 2.92173147e-01 -2.82583348e-0...,-1.29628137e-01 -6.55543581e-02 -7.33080581e-0...,7/4/2008,0.0,1.0
4,3,9.11491811e-02 2.26883858e-01 -5.42985559e-0...,"[19, 5, 1]",13534,-1.19307771e-01 1.82321861e-01 4.61621284e-0...,-8.79967213e-02 3.70598972e-01 -5.36983944e-0...,1/25/2007,0.0,1.0
5,4,2.33093843e-01 -4.37708050e-02 -3.95934820e-0...,"[19, 1, 10]",14398,-3.08268163e-02 4.22684737e-02 -3.10232282e-0...,4.81048003e-02 3.70078124e-02 -3.42290521e-01...,10/8/1997,0.0,1.0
6,2,7.68364891e-02 2.62027904e-02 -4.08470869e-0...,"[12, 4, 7, 8]",9682,-3.06517035e-01 -2.43518919e-01 -2.78039038e-0...,8.84425566e-02 -9.54910293e-02 -8.27141523e-01...,8/11/2006,0.0,1.0
7,19,-1.71428680e-01 5.51107936e-02 -4.86423187e-0...,"[5, 1]",11872,-1.08346120e-01 -4.16447699e-01 -5.08042514e-0...,1.45875802e-03 7.12139383e-02 -3.31807435e-01...,10/19/1962,0.0,1.0
8,5,3.92812304e-02 -1.35863915e-01 -2.89430946e-0...,"[10, 12]",18781,-9.35382470e-02 -3.04444693e-02 -1.96727328e-0...,-9.29433405e-02 -2.03402489e-01 -2.38847956e-0...,3/27/2009,0.0,1.0
9,14,1.07761994e-01 1.39673114e-01 -4.78950143e-0...,"[19, 5, 8]",15836,-3.62595290e-01 -2.57187605e-01 -1.57781005e-0...,4.77287807e-02 1.13167748e-01 -3.90435368e-01...,3/9/2007,0.0,1.0


In [ ]:
"""
print and see the records of train dataset.
"""
train_movies_df.shape[0]

152000

In [ ]:
"""
print distribution of movie genres
"""
all_numbers = [num for sublist in train_movies_df['genres_indices'] for num in sublist]
counts_df = pd.DataFrame(all_numbers, columns=['Number']).groupby('Number').size().reset_index(name='Count')
print(counts_df)

    Number  Count
0        1  22191
1        2   3682
2        3  11751
3        4  13099
4        5  57886
5        6  13359
6        7  10038
7        8  69064
8        9   8420
9       10  30004
10      11   7081
11      12  17500
12      13  25720
13      14   4340
14      15   6433
15      16   5004
16      17  19320
17      18   6261
18      19  27299


In [ ]:
"""
load the vowpal wabbit formatted train dataset
"""
vw_train_data = pd.read_csv(f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/data-prepare/{model}_vw_train_data.txt')

In [ ]:
"""
load the vowpal wabbit model and train it on train data
"""
vw_model = vowpalwabbit.Workspace(f"--cb_explore {len(unique_genres)} --quiet --epsilon 0.1")
for example in vw_train_data:
    vw_model.learn(example)

In [ ]:
"""
save the trained model
"""
vw_model.save(f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/final-results/EP-0.1/{model}_cb_EP-0.1.model')

In [ ]:
"""
this function assigns the cost to the predicted action
"""
def test_assign_cost(predicted_action, movie_id):
    action_context_map = test_movies_df
    search_col = 'id'
    search_val = int(movie_id)
    target_column = 'genres_indices'
    res = action_context_map[(action_context_map[search_col] == search_val)]
# uses the same mathematical formula as stated in the report equation 7
    ind = -1
    index_list = res[target_column].iloc[0]
    if predicted_action in index_list:
      ind = index_list.index(predicted_action)
      if ind == 0:
          return 0
      elif ind >= 1 :
        return ind/len(index_list)
    else:
      return 1

In [ ]:
"""
this function is the probability mass function or pmf.  It takes a list pmf as input, which represents the probabilities of each outcome.
"""
import random
def sample_custom_pmf(pmf):
    total = sum(pmf)
    scale = 1 / total
    pmf = [x * scale for x in pmf]
    draw = random.random()
    sum_prob = 0.0
    for index, prob in enumerate(pmf):
        sum_prob += prob
        if sum_prob > draw:
            return index, prob

In [ ]:
"""
this function is to define the prediction in the model. It passes the action, cost and context in the vowpal wabbit format and triggers the model to make predictions
"""
def predict_genre(row):
    formatted_row = row["keywords_embedding_string"]
    formatted_id = row["id"]
    formatted_original_title_embedding_string = row["original_title_embedding_string"]
    formatted_release_date = row["release_date"]
    formatted_overview_embedding_string = row["overview_embedding_string"]
    context_keywords = f'| feature1:{formatted_id} feature2:{formatted_row} feature3:{formatted_original_title_embedding_string} feature4:{formatted_release_date} feature5:{formatted_overview_embedding_string}'
    action_probabilities = vw_model.predict(context_keywords)
    chosen_index, prob = sample_custom_pmf(action_probabilities)
    return (chosen_index + 1, prob, action_probabilities)

In [ ]:
"""
this function is used to run the prediction iterations.
It stores the results of each iterations in specified files like rewards, regret, accuracy, conversion rate
"""
num_iterations = 300
res = []

totalObtainedRewardsSum = 0
totalObtainedCostSum = 0
iterationRewardsCounter = 0
iterationCostCounter = 0
iterationRewardsList = []
iterationCostList = []
actionCounts = {}
successCounts = {}
accurateCounter = 0

for iteration in range(num_iterations):
  shuffled_indices = np.random.permutation(len(test_movies_df))
  for index in shuffled_indices:
    row = test_movies_df.iloc[index]
    resl = predict_genre(row)
    predicted_genre, prob = resl[0], resl[1]
    keywords_string = row['keywords_embedding_string']
    movie_id = row['id']
    original_title_embedding_string = row["original_title_embedding_string"]
    release_date = row["release_date"]
    overview_embedding_string = row["overview_embedding_string"]
    actual_cost = test_assign_cost(predicted_genre, movie_id)
    totalObtainedRewardsSum += (1-actual_cost)
    iterationRewardsCounter += (1-actual_cost)
    totalObtainedCostSum += actual_cost
    iterationCostCounter += actual_cost

    if predicted_genre not in actionCounts:
      actionCounts[predicted_genre] = 0
    actionCounts[predicted_genre] += 1

    if predicted_genre in row["genres_indices"]:
      accurateCounter += 1
      if predicted_genre not in successCounts:
        successCounts[predicted_genre] = 0
      successCounts[predicted_genre] += 1

    vw_model.learn(f'{predicted_genre}:{actual_cost}:{prob} | feature1:{movie_id} feature2:{keywords_string} feature3:{original_title_embedding_string} feature4:{release_date} feature5:{overview_embedding_string}')

  iterationRewardsList.append(iterationRewardsCounter)
  iterationCostList.append(iterationCostCounter)
  iterationRewardsCounter = 0
  iterationCostCounter = 0

conversion_rates = {}
for action, success_count in successCounts.items():
    total_count = actionCounts[action]
    conversion_rate = success_count / total_count
    conversion_rates[action] = conversion_rate

modelAccuracy = accurateCounter / (len(test_movies_df) * num_iterations)

In [ ]:
output_file_path = f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/final-results/EP-0.1/{model}_res_data_EP-0.1.txt'

with open(output_file_path, 'w') as file:
    for item in res:
        file.write("%s\n" % item)

In [ ]:
"""
print the cumulative rewards over 300 iterations
"""
print(totalObtainedRewardsSum)

6349568.6424702


In [ ]:
"""
print the cumulative regret over 300 iterations
"""
print(totalObtainedCostSum)

5050431.357540556


In [ ]:
"""
print the cumulative rewards after end of each iteration
"""
print(iterationRewardsList)

[9751.034126984134, 12649.24603174599, 13531.15079365073, 14243.026984126904, 14597.088888888822, 15005.641269841175, 15292.778571428486, 15546.48888888881, 15741.476984126886, 16003.0865079364, 16195.496031745928, 16386.067460317317, 16561.362698412577, 16738.207936507835, 16749.062698412592, 17023.535714285586, 17111.074603174515, 17194.31746031734, 17278.16428571416, 17467.730952380884, 17412.310317460222, 17584.80793650788, 17741.284126984046, 17730.560317460233, 17896.088888888782, 18050.817460317383, 18288.5269841269, 18188.81507936504, 18276.20317460311, 18404.272222222182, 18277.360317460272, 18418.200793650707, 18506.843650793562, 18545.929365079337, 18522.676984126945, 18663.50555555551, 18691.507936507867, 18746.188888888842, 18671.85952380947, 18727.16984126976, 19015.812698412643, 18988.881746031722, 19095.467460317337, 19063.198412698373, 19092.160317460268, 19197.21150793647, 19319.79007936505, 19317.16626984124, 19373.711507936463, 19310.083730158694, 19508.70555555552,

In [ ]:
"""
print the cumulative regret after end of each iteration
"""
print(iterationCostList)

[28248.96587301577, 25350.75396825387, 24468.849206349136, 23756.973015872994, 23402.91111111105, 22994.358730158663, 22707.221428571367, 22453.51111111109, 22258.523015872925, 21996.913492063413, 21804.503968253954, 21613.932539682468, 21438.63730158728, 21261.79206349201, 21250.937301587335, 20976.46428571432, 20888.92539682543, 20805.6825396825, 20721.835714285673, 20532.269047619015, 20587.689682539658, 20415.19206349206, 20258.7158730159, 20269.439682539683, 20103.911111111145, 19949.182539682555, 19711.473015873038, 19811.184920634896, 19723.79682539681, 19595.727777777785, 19722.63968253967, 19581.799206349246, 19493.156349206416, 19454.070634920645, 19477.323015873004, 19336.494444444477, 19308.492063492104, 19253.811111111147, 19328.14047619052, 19272.83015873021, 18984.18730158736, 19011.11825396828, 18904.532539682656, 18936.80158730163, 18907.839682539743, 18802.788492063526, 18680.20992063498, 18682.833730158796, 18626.288492063562, 18689.916269841327, 18491.29444444453, 1

In [ ]:
"""
save the cumulative rewards data in a .txt file
"""
output_file_path = f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/final-results/EP-0.1/{model}_plot_reward_data_EP-0.1.txt'

with open(output_file_path, 'w') as file:
    for item in iterationRewardsList:
        file.write("%s\n" % item)

In [ ]:
"""
save the cumulative regret data in a .txt file
"""
output_file_path = f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/final-results/EP-0.1/{model}_plot_cost_data_EP-0.1.txt'

with open(output_file_path, 'w') as file:
    for item in iterationCostList:
        file.write("%s\n" % item)

In [ ]:
"""
save the conversion data in a .txt file
"""
output_file_path = f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/final-results/EP-0.1/{model}_plot_conversion_data_EP-0.1.txt'

with open(output_file_path, 'w') as file:
  for action, conversion_rate in conversion_rates.items():
    file.write("%s %s\n" % (action, conversion_rate))

In [ ]:
"""
save the accuracy data in a .txt file
"""
output_file_path = f'/content/drive/MyDrive/LLM-NEW-2/{model}-embed/final-results/EP-0.1/{model}_plot_accuracy_data_EP-0.1.txt'

with open(output_file_path, 'w') as file:
    file.write("%s \n" % modelAccuracy)

In [ ]:
# End